# Dias et al. (2025): the autumn AHY main model (Table 3), filtering reading (c)

Dataset [DRYAD:bvq83bkk9](https://doi.org/10.5061/dryad.bvq83bkk9), version id 333377 (the only
version; files and SHA-256 in the header of `reproduce_dias_2025.ipynb`). Paper
[doi:10.1098/rspb.2024.2556](https://doi.org/10.1098/rspb.2024.2556).

The hierarchical model of `lm_mod.stan` (Zenodo doi:10.5281/zenodo.14226724) for autumn AHY birds,
as Script01 fits it: scaled wing on scaled tarsus per species-year, with sex effects per species;
the species-year intercepts and slopes depend on PrevPrevPrecipWinMo, PrevTempBreedBase,
PrevPrecipBreedBase, PrevMinTempFallMig, MinTempFallMig and year (per decade); NUTS, adapt_delta
0.85, 4 chains of 4000 iterations with 2000 warm-up, seed 9. Filtering reading (c):
{"a": "the data as deposited", "c": "specimens more than 5 MAD from the median of their species and sex removed",
"d": "specimens more than 5 unscaled MAD from the median of their species removed"}["c"].

Claims tested here, under this reading: table3-autumn-ahy-prior-breeding-temperature,
table3-autumn-ahy-prior-autumn-temperature, table3-autumn-ahy-recent-autumn-temperature and the
main-model part of table3-autumn-ahy-prior-wintering-precipitation. The autumn AHY numbers of claims
that also need the HY and spring AHY models are printed for the record.

In [ ]:
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import wingallom as w

warnings.filterwarnings("ignore", module="rdata")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
import logging

started = time.time()
for name, digest in w.check_staged().items():
    print(f"{name}: SHA-256 {digest} (matches Dryad's file list, version {w.DRYAD_VERSION} of {w.DRYAD_ID})")
t0 = time.time()
print("CmdStan", w.CMDSTAN_VERSION, "at", w.install_cmdstan(Path(tempfile.gettempdir()) / "cmdstan"),
      f"({time.time() - t0:.0f} s to install or find)")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)   # sampler messages; divergences are counted below
code_zip = w.code_zip()
original = code_zip.read("Code/lm_mod.stan").decode()
model_code = w.port_stan(original)
workdir = Path(tempfile.mkdtemp())
t0 = time.time()
model = w.compile_stan(model_code, workdir)
print(f"lm_mod.stan compiled in {time.time() - t0:.0f} s")

READING = "c"

## Data check

In [ ]:
morph = w.morph_prepared(READING)
removed = w.outlier_mask(w.read_csv("morphdata.csv"), READING)
p = w.prepare(morph, "autumn AHY", "main")
data = w.stan_data(p, w.MAIN_PREDICTORS["autumn AHY"])
md_ = p["md"]
print(f"reading ({READING}): {int(removed.sum())} specimens removed by the filter; {len(morph)} kept")
print(f"autumn AHY: {data['N']} birds in {data['J']} species-years of {data['K']} species (paper: 3250 and 171)")
print(f"species-years dropped for a missing climate value: rows {p['missing_rows']} of {p['n_merged']}")
print("\nBirds per species and sex:")
print(pd.crosstab(md_.Taxon, md_.Sex).to_string())
print(f"\nYears {p['G'].year.min()} to {p['G'].year.max()}; year column {data['G'][:, -1].min():.2f} to {data['G'][:, -1].max():.2f} decades")
print(f"scaled wing {data['WL'].min():.2f} to {data['WL'].max():.2f}, scaled tarsus {data['X'][:, 1].min():.2f} to {data['X'][:, 1].max():.2f}")
print("climate columns of G (scaled within species), min and max:")
print(pd.DataFrame(data["G"][:, 1:-1], columns=w.MAIN_PREDICTORS["autumn AHY"][:-1]).agg(["min", "max"]).round(2).to_string())

## Fit

In [ ]:
t0 = time.time()
fit = w.fit_stan(model, data, w.STAN_SEEDS["autumn AHY"], workdir)
fit_minutes = (time.time() - t0) / 60
print(f"fitted in {fit_minutes:.1f} minutes")
print(f"divergent transitions after warm-up: {fit['divergent']} of 8000 draws; largest R-hat over mu_i, mu_s, mu_stl and "
      f"the standard deviations: {fit['max_rhat']:.3f}; smallest bulk ESS: {fit['min_ess_bulk']:.0f}")
table3 = w.effect_table(fit, w.MAIN_PREDICTORS["autumn AHY"])
print("\nTable 3 as fitted here (mean, s.d. and pd of mu_i, effects on the allometric intercept, and mu_s, on the slope):")
print(table3.round(3).to_string())

The sampler does not converge well: 1530 of 8000 draws are divergent and the largest R-hat is 1.188 (smallest bulk ESS 17). The paper's code runs the same model with the same settings and reports no diagnostics, so the estimates below carry this uncertainty.

## table3-autumn-ahy-prior-breeding-temperature

> In autumn AHY birds, warmer temperature in the breeding season of their development (the year before the most recent) is again associated with a more negative allometric intercept (beta -0.09, pd 99.5%) and not with the slope (beta -0.01, pd 75.1%).

In [ ]:
row = table3.loc["PrevTempBreedBase"]
paper = {"int": (-0.09, 0.03, 99.5), "slope": (-0.01, 0.03, 75.1)}
print(f"PrevTempBreedBase, reading ({READING}):")
for part in ("int", "slope"):
    m, sd, pd_ = row[part + "_mean"], row[part + "_sd"], row[part + "_pd"]
    pm, ptol, ppd = paper[part]
    print(f"  {'intercept' if part == 'int' else 'slope':9s} mean {m:+.3f} (s.d. {sd:.3f}), pd {pd_:.1%};"
          f" paper {pm:+.2f} (tolerance {ptol}), pd {ppd}%; within tolerance: {abs(m - pm) <= ptol + 1e-9};"
          f" sign as paper: {np.sign(m) == np.sign(pm) or pm == 0}; pd >= 90%: {pd_ >= 0.9}")

**Result: under reading (c), the claim holds.** Intercept -0.091 (pd 99.4%), slope -0.026 (pd 76.5%), within
tolerance.

## table3-autumn-ahy-prior-autumn-temperature

> In autumn AHY birds, the temperature of the autumn migration before the most recent one has no significant association with the allometric intercept (beta -0.02, pd 78.0%) or slope (beta 0.01, pd 70.9%).

In [ ]:
row = table3.loc["PrevMinTempFallMig"]
paper = {"int": (-0.02, 0.07, 78.0), "slope": (0.01, 0.05, 70.9)}
print(f"PrevMinTempFallMig, reading ({READING}):")
for part in ("int", "slope"):
    m, sd, pd_ = row[part + "_mean"], row[part + "_sd"], row[part + "_pd"]
    pm, ptol, ppd = paper[part]
    print(f"  {'intercept' if part == 'int' else 'slope':9s} mean {m:+.3f} (s.d. {sd:.3f}), pd {pd_:.1%};"
          f" paper {pm:+.2f} (tolerance {ptol}), pd {ppd}%; within tolerance: {abs(m - pm) <= ptol + 1e-9};"
          f" sign as paper: {np.sign(m) == np.sign(pm) or pm == 0}; pd >= 90%: {pd_ >= 0.9}")

**Result: under reading (c), the claim holds.** Intercept -0.019 (pd 74.2%), slope 0.008 (pd 63.3%).

## table3-autumn-ahy-recent-autumn-temperature

> In autumn AHY birds, temperature during the autumn migration in which they were collected has no significant association with the allometric intercept (beta 0.03, pd 88.6%) but a negative association with the allometric slope (beta -0.03, pd 90.3%).

In [ ]:
row = table3.loc["MinTempFallMig"]
paper = {"int": (0.03, 0.04, 88.6), "slope": (-0.03, 0.05, 90.3)}
print(f"MinTempFallMig, reading ({READING}):")
for part in ("int", "slope"):
    m, sd, pd_ = row[part + "_mean"], row[part + "_sd"], row[part + "_pd"]
    pm, ptol, ppd = paper[part]
    print(f"  {'intercept' if part == 'int' else 'slope':9s} mean {m:+.3f} (s.d. {sd:.3f}), pd {pd_:.1%};"
          f" paper {pm:+.2f} (tolerance {ptol}), pd {ppd}%; within tolerance: {abs(m - pm) <= ptol + 1e-9};"
          f" sign as paper: {np.sign(m) == np.sign(pm) or pm == 0}; pd >= 90%: {pd_ >= 0.9}")

**Result: under reading (c), the claim holds.** Intercept 0.027 (pd 88.9%), slope -0.032 (pd 91.2%), within
tolerance.

## table3-autumn-ahy-prior-wintering-precipitation

> In autumn AHY birds the HY association does not hold. Wintering precipitation before development has a positive association with the allometric intercept (beta 0.03, pd 90.0%), none with the slope (beta 0.02, pd 88.7%), and pre-breeding precipitation before development is not associated with wing length in the preliminary model (Table S6). (The preliminary part is in `reproduce_dias_2025.ipynb`.)

In [ ]:
row = table3.loc["PrevPrevPrecipWinMo"]
paper = {"int": (0.03, 0.03, 90.0), "slope": (0.02, 0.03, 88.7)}
print(f"PrevPrevPrecipWinMo, reading ({READING}):")
for part in ("int", "slope"):
    m, sd, pd_ = row[part + "_mean"], row[part + "_sd"], row[part + "_pd"]
    pm, ptol, ppd = paper[part]
    print(f"  {'intercept' if part == 'int' else 'slope':9s} mean {m:+.3f} (s.d. {sd:.3f}), pd {pd_:.1%};"
          f" paper {pm:+.2f} (tolerance {ptol}), pd {ppd}%; within tolerance: {abs(m - pm) <= ptol + 1e-9};"
          f" sign as paper: {np.sign(m) == np.sign(pm) or pm == 0}; pd >= 90%: {pd_ >= 0.9}")

**Result: under reading (c), the main-model part falls just short.** Intercept 0.029 (pd 89.1%, below 90%),
slope 0.020 (pd 81.5%); signs and means hold.

## Autumn AHY parts of claims that also need the HY and spring AHY models

These claims are blocked for time (the HY and spring AHY fits are not run); the autumn AHY numbers
are printed for the record: year (tables1-3-year-intercept, tables2-3-ahy-year-slope-none) and
breeding precipitation before development (tables1-3-breeding-precipitation).

In [ ]:
print(table3.loc[["y_id", "PrevPrecipBreedBase"]].round(3).to_string())
print("paper, Table 3: year intercept 0.19 (pd 99.8%), slope 0.02 (pd 75.9%); PrevPrecipBreedBase intercept -0.05 (pd 93.4%), slope -0.01 (pd 65.3%)")
print(f"\nNotebook time: {(time.time() - started) / 60:.1f} minutes")